# Task III: Open Task — Commentary on Quantum Machine Learning

**Author:** Lakshikka Sithamparanathan  
**Affiliation:** PhD Student, Computer Science, Cleveland State University  

---

This commentary draws from my active PhD research in quantum machine learning, 
which spans quantum kernel methods, quantum contrastive learning, and hybrid 
quantum-classical architectures. Rather than summarising textbook material, 
I focus on observations and open questions that have emerged directly from 
designing and running experiments.

## 1. What Quantum Machine Learning Is and What It Isn't

One of the most important realisations in my research has been understanding 
the difference between quantum-*inspired* and genuinely quantum machine learning. 
Many methods labelled as QML use classical simulators with exponential overhead - 
they simulate Hilbert space on classical hardware without any actual quantum advantage. 
True QML requires thinking carefully about *where* quantum mechanics provides 
a representational or computational advantage that cannot be efficiently replicated 
classically.

In my view, the most promising avenue is not replacing classical neural networks 
with quantum circuits wholesale, but identifying the *specific subroutines* 
where quantum geometry provides an inductive bias that is physically meaningful 
for the problem at hand. For high energy physics data, where jets involve 
multi-particle correlations with Lorentz symmetry and IRC-safe structure, 
the exponential dimensionality of Hilbert space may encode these correlations 
more naturally than a flat Euclidean embedding space.

## 2. Quantum Kernel Methods: What My Research Has Found

A significant portion of my PhD research involves quantum kernel bandwidth 
optimisation. The core insight from Schuld and Killoran (2019) is that a 
parameterised quantum circuit defines a kernel:

$$K(x_i, x_j; \theta) = |\langle \psi(x_i; \theta) | \psi(x_j; \theta) \rangle|^2$$

This kernel can be used with a classical SVM — no quantum measurement 
statistics are needed beyond the pairwise fidelity values.

In 630 experiments across three datasets (MNIST, Fashion-MNIST, KMNIST) 
and three kernel families (Pauli, ZZ, IQP), I have found a consistent pattern: 
**the bandwidth of the data encoding how aggressively features are scaled 
before encoding as rotation angles has a larger impact on classification 
accuracy than the choice of entangling structure**. Specifically, IQP kernels 
with features scaled to $[0, 2\pi]$ systematically underperform the same 
architecture with features scaled to $[0, \pi/2]$, because over-rotation 
causes the kernel values to cluster near 0.5 regardless of class — 
a quantum analogue of the kernel concentration phenomenon in high dimensions.

This finding has a direct practical implication for the QMLHEP6 project: 
before training a contrastive embedding, it is worth spending time on 
empirical kernel alignment — checking whether the initial (random-weight) 
quantum kernel has any signal at all for the HEP classification task. 
If the untrained kernel is already random, gradient-based training is unlikely 
to recover structure.

## 3. Data Re-Uploading: The Method I Find Most Promising

The **data re-uploading** framework of Pérez-Salinas et al. (Quantum 4, 226, 2020) 
is the approach I have spent the most time implementing and extending. 
The key insight is that a quantum circuit with a single encoding layer 
can only access a limited Fourier spectrum of the input data — the number 
of accessible frequency components grows with the number of re-uploading 
repetitions.

Concretely, for a circuit with $L$ data re-uploading layers where each 
layer encodes feature $x_i$ via $R_Y(x_i)$, the output state can represent 
functions with Fourier terms up to order $L$. This means expressivity is 
*directly controllable* — adding a layer is equivalent to adding a higher-order 
Fourier term. This is a much more transparent expressivity-depth tradeoff 
than classical neural networks, where adding a layer has complex effects 
on the learned function class.

In Task VI, I implemented data re-uploading with independent single-qubit 
rotation gates as the trainable component of each layer. Each layer applies:
1. `RY(x_i)` — re-encode the data at each qubit $i$
2. `RX(θ_{l,i,0})`, `RY(θ_{l,i,1})`, `RZ(θ_{l,i,2})` — trainable rotations 
   covering the full SU(2) space per qubit
3. Ring CNOT entanglement — `CNOT(i, (i+1) % n)` for all qubits

This gives 3 × n_qubits trainable parameters per layer (72 total across 3 layers 
and 8 qubits), with a fixed ring entanglement topology. The three-axis rotation 
structure ensures full SU(2) coverage per qubit, which the Fourier expressivity 
argument requires for the circuit to access the full frequency spectrum up to 
order $L = 3$.

## 4. Methods I Would Like to Work On

### 4.1 Equivariant Quantum Circuits for HEP Symmetries

The most exciting open direction in my view is building **Lorentz-equivariant 
quantum circuits** for jet physics. Classical equivariant networks (LorentzNet, 
PELICAN) achieve state-of-the-art performance on top quark tagging precisely 
because they hard-code Lorentz invariance as an architectural constraint — 
the network cannot learn non-physical representations even in principle.

The quantum analogue would be a parameterised circuit whose unitary evolution 
commutes with Lorentz transformations of the input features. Meyer et al. 
(arXiv: 2205.06217) have shown how to construct $G$-equivariant quantum circuits 
for discrete groups; extending this to the continuous Lorentz group is an 
open problem that I believe would significantly improve quantum jet classifiers.

### 4.2 Quantum Attention for Set-Structured Data

My survey on quantum attention mechanisms (under review) 
identified a gap: existing quantum attention proposals focus on sequence-structured 
data (text, time series) but jets are *sets* of particles with no natural ordering. 
A permutation-invariant quantum attention mechanism — where the query-key 
similarity is computed via SWAP-test fidelity rather than dot products — 
would be a natural fit for jet physics and would address a genuine 
architectural limitation in current quantum transformers.

### 4.3 Quantum Contrastive Learning with Physics-Aware Augmentations

The most direct extension of my current work (Task VI, QMLHEP6) is developing 
**IRC-safe quantum augmentations** — transformations of jet data that preserve 
physics symmetries (infrared and collinear safety) while generating diverse 
views for self-supervised contrastive training. JetCLR (Dillon et al., 2022) 
showed that classical contrastive learning on jets dramatically improves 
when augmentations respect QCD symmetries rather than using generic image 
augmentations. Implementing the quantum analogue — where augmented views 
produce quantum states that should have high fidelity by construction — 
is the natural next step for the QMLHEP6 project.

## 5. Honest Assessment: Where Quantum ML Stands Today

I want to be direct about something that is often glossed over in QML papers: 
**we do not yet have convincing evidence of practical quantum advantage 
in machine learning on near-term devices**. Classical simulation can match 
or exceed most current quantum ML results, and the exponential scaling 
advantage of quantum computing only manifests at qubit counts far beyond 
what current hardware supports with acceptable error rates.

However, I believe this is the right moment to do QML research for three reasons:

1. **The algorithmic foundations need to be built now.** When fault-tolerant 
   quantum hardware arrives, the community will need ready-made quantum 
   algorithms for scientific data analysis. The work being done today on 
   quantum kernels, quantum contrastive learning, and equivariant circuits 
   is building that foundation.

2. **The low-parameter regime is genuinely interesting.** QRGCL (Jahin et al., 2024) 
   achieved competitive jet discrimination with only 45 trainable quantum parameters — 
   a regime where classical models require thousands. This parameter efficiency 
   matters for online trigger systems where model update speed is a hard constraint.

3. **Hilbert space geometry provides a novel representational language.** 
   Even setting aside computational speedup, the geometry of quantum state space — 
   complex projective space with its Fubini-Study metric and entanglement structure — 
   may capture physical correlations in HEP data more naturally than Euclidean 
   embeddings. Whether it actually does is an empirical question, 
   and answering it rigorously is the goal of QMLHEP6.

The honest path forward is careful, reproducible benchmarking — which is 
precisely what I propose to do.